In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough 
from langchain.callbacks import StreamingStdOutCallbackHandler


llm = ChatOpenAI(temperature=0.1,
                 streaming=True,
                 callbacks=[StreamingStdOutCallbackHandler()])

cache_dir = LocalFileStore("../../.cache")

memory = ConversationSummaryBufferMemory(
    llm=llm,
    return_messages=True,
    memory_key="chat_history",
    # output_key="answer"
)

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100
)

loader = UnstructuredFileLoader("../../files/chapter_03.txt")

docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    memory=memory,
    chain_type="stuff",
    verbose=True
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "다음 내용을 토대로 답변하세요, {context}"),
        ("human", "{question}")
    ]
)

retriever = vectorstore.as_retriever()

chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

chain.invoke("Aaronson은 유죄인가요?")
chain.invoke("그가 테이블에 어떤 메시지를 썼나요?")
chain.invoke("Julia 는 누구인가요?")


Created a chunk of size 717, which is longer than the specified 600
Created a chunk of size 608, which is longer than the specified 600
Created a chunk of size 642, which is longer than the specified 600
Created a chunk of size 1444, which is longer than the specified 600
Created a chunk of size 1251, which is longer than the specified 600
Created a chunk of size 1012, which is longer than the specified 600
Created a chunk of size 1493, which is longer than the specified 600
Created a chunk of size 819, which is longer than the specified 600
Created a chunk of size 1458, which is longer than the specified 600
Created a chunk of size 1411, which is longer than the specified 600
Created a chunk of size 742, which is longer than the specified 600
Created a chunk of size 669, which is longer than the specified 600
Created a chunk of size 906, which is longer than the specified 600
Created a chunk of size 703, which is longer than the specified 600
Created a chunk of size 1137, which is lon

Aaronson은 유죄로 판결받았습니다.그가 테이블에 쓴 메시지는 "'Under the spreading chestnut tree I sold you and you sold me----'" 입니다.주어진 문맥에서 Julia는 조지 오웰의 소설 '1984'의 주인공 윈스턴 스미스(Winston Smith)가 사랑하는 여성 캐릭터입니다. 이 소설에서 Julia는 윈스턴의 사랑이자 동료인데, 빅 브라더(Big Brother)의 단속과 감시 속에서 금기된 사랑을 함께 경험하게 됩니다. Julia는 윈스턴에게 희망과 사랑을 상징하는 캐릭터로 등장합니다.

AIMessageChunk(content="주어진 문맥에서 Julia는 조지 오웰의 소설 '1984'의 주인공 윈스턴 스미스(Winston Smith)가 사랑하는 여성 캐릭터입니다. 이 소설에서 Julia는 윈스턴의 사랑이자 동료인데, 빅 브라더(Big Brother)의 단속과 감시 속에서 금기된 사랑을 함께 경험하게 됩니다. Julia는 윈스턴에게 희망과 사랑을 상징하는 캐릭터로 등장합니다.")